In [1]:
import os
import json
import time
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, datasets, models
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score

# --- SCOTT'S LOGGING UTILITY ---
def save_training_log(model_name, hyperparameters, metrics, observations, student_name="ScottL"):
    log_dir = "logs"
    os.makedirs(log_dir, exist_ok=True)
    log_entry = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "student": student_name,
        "model_backbone": model_name,
        "hyperparameters": hyperparameters,
        "results": metrics,
        "observations": observations
    }
    filename = f"{log_dir}/{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(filename, 'w') as f:
        json.dump(log_entry, f, indent=4)
    print(f"✅ Log saved to GitHub history: {filename}")

In [2]:
def get_data_loaders(data_root, batch_size=32):
    # Standard transforms for WaRP-C
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_dataset = datasets.ImageFolder(root=os.path.join(data_root, 'train'), transform=transform)
    
    # --- FIX: CLASS IMBALANCE ---
    # Calculate weights: 1 / frequency of each class
    targets = np.array(train_dataset.targets)
    class_sample_count = np.array([len(np.where(targets == t)[0]) for t in np.unique(targets)])
    weight = 1. / class_sample_count
    samples_weight = torch.from_numpy(weight[targets]).double()
    
    # The sampler ensures the model sees a balanced distribution during training
    sampler = WeightedRandomSampler(samples_weight, len(samples_weight))

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=2)
    
    # Validation and Test don't need samplers
    val_dataset = datasets.ImageFolder(root=os.path.join(data_root, 'val'), transform=transform)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, train_dataset.classes

In [3]:
def get_model(model_name, num_classes=28):
    if model_name == "resnet50":
        model = models.resnet50(weights='IMAGENET1K_V1')
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "efficientnet_b3":
        model = models.efficientnet_b3(weights='IMAGENET1K_V1')
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == "swin_t":
        model = models.swin_t(weights='IMAGENET1K_V1')
        model.head = nn.Linear(model.head.in_features, num_classes)
    elif model_name == "convnext_t":
        model = models.convnext_tiny(weights='IMAGENET1K_V1')
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    else:
        raise ValueError("Model not supported in this factory.")
    
    return model

In [4]:
def train_model(model, train_loader, val_loader, epochs=10, lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        # ... standard training loop logic ...
        print(f"Epoch {epoch+1} complete.")

    # --- FINAL EVALUATION (MANDATORY METRICS) ---
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    # Calculate mandatory metrics for the report
    f1 = f1_score(all_labels, all_preds, average='macro')
    auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
    map_score = average_precision_score(torch.get_dummy(torch.tensor(all_labels), 28), all_probs)

    results = {"F1": f1, "AUC": auc, "mAP": map_score}
    return results